# ARCHS4 CLAMP models: Effect of CLAMP_K (10% study-based sampling)

**Environment:** `clamp-analyses`

This notebook evaluates the effect of different CLAMP_K values on model results using 10% study-based sampling of ARCHS4 data.

Instead of estimating CLAMP_K via `num.pc()`, we fix it to: 250, 500, 1250, 2500, 3750, 5000 (same as 5% notebook).

For each CLAMP_K value, 5 models are built with different random seeds (and thus different subsamples and SVDs).

Structure per seed:
1. Subsample ARCHS4 data (10% coverage, study-based shuffling)
2. Compute SVD (shared across CLAMP_K values for the same seed)
3. For each CLAMP_K: run CLAMPbase, then CLAMPfull with BP prior

## Load libraries

In [1]:
if (!requireNamespace("CLAMP", quietly = TRUE)) {
    REPO_PATH <- "/home/msubirana/Documents/pivlab/CLAMP" 
    remotes::install_local(REPO_PATH, force = TRUE, dependencies = FALSE)
}

library(bigstatsr)
library(data.table)
library(dplyr)
library(rsvd)
library(glmnet)
library(Matrix)
library(knitr)
library(here)
library(CLAMP)
library(rhdf5)

source(here("config.R"))


Attaching package: ‘dplyr’


The following objects are masked from ‘package:data.table’:

    between, first, last


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: Matrix

Loaded glmnet 4.1-10

here() starts at /home/msubirana/Documents/pivlab/clamp-analyses



## Configuration

In [2]:
# Base output directory
base_output_dir <- config$ARCHS4$DATASET_FOLDER

# Coverage level
coverage <- 0.10
coverage_pct <- coverage * 100

# CLAMP_K values to test (same as 5% notebook)
clamp_k_values <- c(250, 500, 1250, 2500, 3750, 5000)

# Data path for priors
data_path <- here::here('data/archs4')
archs4_file <- here('data/archs4/human_gene_v2.5.h5')

# SVD parameters
N_CORES <- 4

# Seeds for 5 runs
base_seed <- config$ARCHS4$CLAMP_PARAMS$RANDOM_SVD_SEED
seeds <- base_seed + 0:4
n_runs <- length(seeds)

# Output directory
output_base <- file.path(base_output_dir, "clamp_k_effect_10_study")
dir.create(output_base, showWarnings = FALSE, recursive = TRUE)

message("CLAMP_K values: ", paste(clamp_k_values, collapse = ", "))
message("Seeds: ", paste(seeds, collapse = ", "))
message("Coverage: ", coverage_pct, "%")
message("Output: ", output_base)

CLAMP_K values: 250, 500, 1250, 2500, 3750, 5000

Seeds: 123, 124, 125, 126, 127

Coverage: 10%

Output: /home/msubirana/Documents/pivlab/clamp-analyses/output/archs4/clamp_k_effect_10_study



## Load preprocessed data

In [3]:
# Load metadata
meta <- readRDS(file.path(base_output_dir, "metadata_filtered.rds"))
n_genes_thin <- meta$n_genes_thin
n_samples_total <- meta$n_samples
archs4_genes <- meta$gene_symbols_thin

# Load sample names
all_samples <- readRDS(file.path(base_output_dir, "all_samples.rds"))
sample_names_total <- all_samples[seq_len(n_samples_total)]

# Load BP pathway
BP_pathMat <- readRDS(file.path(data_path, "BP_pathMat.rds"))
BP_matched <- getMatchedPathwayMat(BP_pathMat, archs4_genes)
message("Loaded and matched BP pathway matrix")

# Load FBM
fbm_file <- file.path(base_output_dir, "fbm")
output_file <- paste0(fbm_file, "_filtered")

archs4_fbm_filt <- FBM(
  nrow        = n_genes_thin,
  ncol        = n_samples_total,
  backingfile = output_file,
  create_bk   = FALSE
)

message("Loaded FBM with ", n_genes_thin, " genes and ", n_samples_total, " samples")

# Get series information from ARCHS4 HDF5 file
message("Getting series information from ARCHS4 HDF5 file...")

geo_acc <- h5read(archs4_file, "/meta/samples/geo_accession")
series_id <- h5read(archs4_file, "/meta/samples/series_id")

gsm_to_gse <- data.frame(
  geo_accession = geo_acc,
  series_id = series_id,
  stringsAsFactors = FALSE
)

# Create mapping of our filtered samples to series
sample_series_df <- data.frame(
  sample_idx = seq_len(n_samples_total),
  sample_name = sample_names_total,
  stringsAsFactors = FALSE
) %>%
  left_join(gsm_to_gse, by = c("sample_name" = "geo_accession"))

# Get unique series and their sample counts
series_counts <- sample_series_df %>%
  group_by(series_id) %>%
  summarise(n_samples = n(), .groups = "drop") %>%
  arrange(series_id)

message("Found ", nrow(series_counts), " unique studies")

There are 11936 genes in the intersection between data and prior

Removing 2059 pathways

Loaded and matched BP pathway matrix

Loaded FBM with 18423 genes and 605614 samples

Getting series information from ARCHS4 HDF5 file...

Found 18084 unique studies



## Run models: 5 seeds x 6 CLAMP_K values

For each seed: study-based subsample → SVD (once), then loop over CLAMP_K values for CLAMPbase + CLAMPfull_BP.

In [ ]:
n_samples_target <- round(n_samples_total * coverage)

SVD_K <- min(n_samples_target - 1, n_genes_thin - 1)

message("Target samples per run: ", n_samples_target, " (", coverage_pct, "% of ", n_samples_total, ")")
message("SVD K: ", SVD_K)

results_summary <- data.frame(
  run = integer(),
  seed = integer(),
  CLAMP_K = integer(),
  n_samples = integer(),
  n_studies = integer(),
  stringsAsFactors = FALSE
)

for (run_idx in seq_len(n_runs)) {
  current_seed <- seeds[run_idx]
  message("\n", strrep("=", 60))
  message("SEED ", run_idx, "/", n_runs, " (", current_seed, ")")
  message(strrep("=", 60))
  
  seed_dir <- file.path(output_base, paste0("seed_", run_idx))
  dir.create(seed_dir, showWarnings = FALSE, recursive = TRUE)
  
  # --- Study-based sample selection ---
  set.seed(current_seed)
  shuffled_series <- sample(series_counts$series_id)
  
  selected_samples <- c()
  selected_series <- c()
  n_collected <- 0
  
  for (s in shuffled_series) {
    if (n_collected >= n_samples_target) break
    series_samples <- sample_series_df$sample_idx[sample_series_df$series_id == s]
    n_series_samples <- length(series_samples)
    
    if (n_series_samples == 0) {
      warning("No samples found for series: ", s)
      next
    }
    
    if (n_collected + n_series_samples <= n_samples_target) {
      selected_samples <- c(selected_samples, series_samples)
      selected_series <- c(selected_series, s)
      n_collected <- n_collected + n_series_samples
    } else {
      n_needed <- n_samples_target - n_collected
      selected_samples <- c(selected_samples, series_samples[1:n_needed])
      selected_series <- c(selected_series, s)
      n_collected <- n_collected + n_needed
      break
    }
  }
  
  sample_idx <- sort(selected_samples)
  n_samples <- length(sample_idx)
  sample_names <- sample_names_total[sample_idx]
  
  message("Selected ", length(selected_series), " studies with ", n_samples, " total samples")
  
  saveRDS(list(
    run = run_idx,
    seed = current_seed,
    coverage = coverage,
    n_samples = n_samples,
    sample_idx = sample_idx,
    sample_names = sample_names,
    selected_series = selected_series,
    sampling_method = "study_based"
  ), file = file.path(seed_dir, "subsample_info.rds"))
  
  # --- Create subsampled FBM ---
  message("Creating subsampled FBM...")
  fbm_sub_file <- file.path(seed_dir, "fbm_subsampled")
  
  Y_sub <- big_copy(
    archs4_fbm_filt,
    ind.col = sample_idx,
    backingfile = fbm_sub_file
  )
  
  # --- SVD (once per seed) ---
  message("Computing SVD (k=", SVD_K, ")...")
  
  if (N_CORES > 1) {
    options(bigstatsr.check.parallel.blas = FALSE)
    blas_nproc <- getOption("default.nproc.blas")
    options(default.nproc.blas = NULL)
  }
  
  svd_result <- big_randomSVD(Y_sub, k = SVD_K, ncores = N_CORES)
  
  if (N_CORES > 1) {
    options(bigstatsr.check.parallel.blas = TRUE)
    options(default.nproc.blas = blas_nproc)
  }
  
  valid_idx <- which(!is.nan(svd_result$d))
  svd_result$d <- svd_result$d[valid_idx]
  svd_result$u <- svd_result$u[, valid_idx, drop = FALSE]
  svd_result$v <- svd_result$v[, valid_idx, drop = FALSE]
  
  saveRDS(svd_result, file = file.path(seed_dir, "svd.rds"))
  message("SVD done: ", length(svd_result$d), " valid components")
  
  # --- Loop over CLAMP_K values ---
  for (CLAMP_K in clamp_k_values) {
    message("\n--- CLAMP_K = ", CLAMP_K, " (seed ", run_idx, ") ---")
    
    k_dir <- file.path(seed_dir, paste0("clamp_k_", CLAMP_K))
    dir.create(k_dir, showWarnings = FALSE, recursive = TRUE)
    
    saveRDS(CLAMP_K, file = file.path(k_dir, "CLAMP_K.rds"))
    
    # CLAMPbase
    message("Running CLAMPbase...")
    baseRes <- CLAMPbase(
      Y = Y_sub,
      svdres = svd_result,
      trace = TRUE,
      clamp_k = CLAMP_K
    )
    
    baseRes$Z <- data.frame(baseRes$Z)
    rownames(baseRes$Z) <- archs4_genes
    baseRes$B <- data.frame(baseRes$B)
    colnames(baseRes$B) <- sample_names
    
    saveRDS(baseRes, file = file.path(k_dir, "CLAMPbase.rds"))
    
    model_dir <- file.path(k_dir, "CLAMPbase")
    dir.create(model_dir, showWarnings = FALSE, recursive = TRUE)
    write.csv(baseRes$B, file.path(model_dir, "B.csv"))
    write.csv(baseRes$Z, file.path(model_dir, "Z.csv"))
    
    # CLAMPfull with BP prior
    message("Running CLAMPfull with BP prior...")
    fullRes <- CLAMPfull(
      Y = Y_sub,
      svdres = svd_result,
      priorMat = BP_matched,
      clamp.base.result = baseRes,
      use_cpp = TRUE,
      trace = TRUE,
      clamp_k = CLAMP_K
    )
    
    fullRes$Z <- data.frame(fullRes$Z)
    rownames(fullRes$Z) <- archs4_genes
    fullRes$B <- data.frame(fullRes$B)
    colnames(fullRes$B) <- sample_names
    fullRes$summary <- fullRes$summary %>%
      dplyr::rename(LV = LV_index) %>%
      dplyr::mutate(LV = paste0('LV', LV))
    
    saveRDS(fullRes, file = file.path(k_dir, "CLAMPfull_BP.rds"))
    
    model_dir <- file.path(k_dir, "CLAMPfull_BP")
    dir.create(model_dir, showWarnings = FALSE, recursive = TRUE)
    write.csv(fullRes$B, file.path(model_dir, "B.csv"))
    write.csv(fullRes$Z, file.path(model_dir, "Z.csv"))
    write.csv(fullRes$summary, file.path(model_dir, "summary.csv"))
    
    # Store summary
    results_summary <- rbind(results_summary, data.frame(
      run = run_idx,
      seed = current_seed,
      CLAMP_K = CLAMP_K,
      n_samples = n_samples,
      n_studies = length(selected_series)
    ))
    
    rm(baseRes, fullRes)
    gc()
  }
  
  rm(Y_sub, svd_result)
  gc()
}

message("\n", strrep("=", 60))
message("All runs completed!")
message(strrep("=", 60))

Target samples per run: 60561 (10% of 605614)

SVD K: 18422



SEED 1/5 (123)


Selected 2237 studies with 60561 total samples

Creating subsampled FBM...

Computing SVD (k=18422)...



## Summary

In [ ]:
# Save summary
write.csv(results_summary, file.path(output_base, "results_summary.csv"), row.names = FALSE)

kable(results_summary)
message("Results saved to: ", output_base)